# SageAgent v5 Production UI

SageAgent v5 is the production-ready SageMaker notebook coding agent. It keeps the v4-style notebook UI, but runs the rebuilt v5 engine underneath: Bedrock Claude models, durable memory/status, subagents, checkpoints, compaction, result replay, telemetry, and verify/done gates.

**Run cells 1-2 in order:**
- **Cell 1** installs non-widget packages, usually once per kernel. It deliberately does not install or upgrade `ipywidgets`, which is more conservative than v4 and avoids browser/kernel widget-version drift.
- **Cell 2** launches the single combined UI. Configure model, workspace, budgets, thinking, mock mode, Bedrock-only, approval, and chat from that one UI.

**Core runtime files in the production zip:** `chat.ipynb`, `entry.py`, `sagemaker_agent.py`, `agent.py`, `commands.py`, `memory.md`, `AGENT_STATUS.md`, and the `core/`, `runtime/`, `tools/`, `prompt/`, `skills/`, `subagent/`, `security/`, and `ui/` packages.

**Companion guide:** open `chat.md` for the user guide, command list, skills list, safety notes, and troubleshooting.


In [ ]:
# Cell 1: install non-widget dependencies (run once per kernel)
# Do not install/upgrade ipywidgets here. SageMaker's browser-side widget
# manager must match the environment it already provides; changing the
# kernel widget package can cause browser-side widget model failures.
!pip install -q boto3 Pillow python-docx pandas openpyxl matplotlib requests scikit-learn

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError(
        "ipywidgets is not available in this kernel. Use a SageMaker/Jupyter "
        "image with ipywidgets already installed, or install a version that "
        "matches the browser-side widget manager before launching v5."
    ) from exc

print(f"ipywidgets already available: {widgets.__version__}")


In [ ]:
# Compatibility test: old refresh cell shape from stale notebooks.
import sys
from pathlib import Path
from IPython.display import clear_output

clear_output(wait=True)
for _p in [Path.cwd(), Path.cwd() / "compact_v5", Path.cwd() / "sagemaker-coding-agent" / "compact_v5"]:
    if (_p / "entry.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
for _m in ["ui.chat_ui", "ui.widgets", "entry"]:
    sys.modules.pop(_m, None)
from entry import refresh_ui
ui = refresh_ui(use_widgets=True)
